# 03 — Mock Simulation Reproduces Historical Inventory

`DataLoaderMock` generates trips first, then reconstructs hourly inventory deterministically from those trips. Under that construction, a simulator run with **only** `ArrivalsPhase` + `DemandPhase` (in that order, organic supply seeded into `in_transit` by `init_state`) must reproduce `resolved.observed_inventory` **exactly** at daily grain.

This notebook verifies that invariant end-to-end.

In [ ]:
import pandas as pd

from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 168,  # 7 days x 24 hours
    "time_freq": "h",
    "start_date": "2025-01-01",
    "num_resources": 3,
    "resource_capacity": 100,
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}

mock = DataLoaderMock(mock_config)
loader = DataLoaderGraph(mock, GraphLoaderConfig())
loader.load_data()
resolved = loader.resolved

## 1. Per-day flow invariant on raw mock inventory

For every day `D`, every station, every commodity:
`inv_ts[last_hour_of_D] == inv_ts[first_hour_of_D] + arrivals(D) − departures(D)`

where arrivals/departures are grouped by `started_at.dt.date` (valid because the generator forces every trip to be strictly intra-day).

In [ ]:
inv_ts = mock.df_inventory_ts
trips = mock.df_trips.copy()
trips["date"] = trips["started_at"].dt.normalize()

cross_day_trips = (
    trips["started_at"].dt.normalize() != trips["ended_at"].dt.normalize()
).sum()
first_hour_starts = (trips["started_at"].dt.hour == 0).sum()

ts_index = pd.DatetimeIndex(inv_ts.index)
normalized_days = ts_index.normalize()
unique_days = pd.DatetimeIndex(pd.unique(normalized_days))
day_first_hour = [ts_index[normalized_days == day][0] for day in unique_days]
day_last_hour = [ts_index[normalized_days == day][-1] for day in unique_days]

station_ids = mock.df_stations["station_id"].tolist()
rows = []
for day_first, day_last in zip(day_first_hour, day_last_hour, strict=True):
    day = day_first.normalize()
    day_trips = trips[trips["date"] == day]
    for station_id in station_ids:
        for commodity in ("electric_bike", "classic_bike"):
            col = inv_ts[(station_id, commodity)]
            commodity_trips = day_trips[day_trips["rideable_type"] == commodity]
            departures = int((commodity_trips["start_station_id"] == station_id).sum())
            arrivals = int((commodity_trips["end_station_id"] == station_id).sum())
            rows.append({
                "day": day,
                "facility_id": station_id,
                "commodity_category": commodity,
                "start_of_day": int(col.loc[day_first]),
                "end_of_day": int(col.loc[day_last]),
                "departures": departures,
                "arrivals": arrivals,
            })
per_day_invariant = pd.DataFrame(rows)
per_day_invariant["expected_end"] = (
    per_day_invariant["start_of_day"]
    + per_day_invariant["arrivals"]
    - per_day_invariant["departures"]
)
per_day_invariant["diff"] = per_day_invariant["end_of_day"] - per_day_invariant["expected_end"]
invariant_max_abs_diff = per_day_invariant["diff"].abs().max()
invariant_violations = int((per_day_invariant["diff"].abs() > 0).sum())

assert cross_day_trips == 0, f"generator produced {cross_day_trips} cross-day trips"
assert first_hour_starts == 0, f"generator produced {first_hour_starts} first-hour trip starts"
assert invariant_violations == 0, (
    f"per-day invariant violated in {invariant_violations} rows "
    f"(max abs diff {invariant_max_abs_diff})"
)

## 2. Simulator reproduces `observed_inventory` exactly

Run `Environment` with phases `[ArrivalsPhase, DemandPhase]`. Organic returns are seeded from `resolved.supply` into `state.in_transit` by `init_state`, so `ArrivalsPhase` picks them up in each period. Compare the end-of-period inventory snapshot against `resolved.observed_inventory` (which is "last-telemetry-of-day" per station/commodity).

In [ ]:
from gbp.consumers.simulator import Environment, EnvironmentConfig
from gbp.consumers.simulator.built_in_phases import ArrivalsPhase, DemandPhase

env_config = EnvironmentConfig(
    phases=[ArrivalsPhase(), DemandPhase()],
    seed=42,
    scenario_id="reproduce_history",
)
env = Environment(resolved, env_config)
simulation_log = env.run().to_dataframes()
sim_inventory = simulation_log["simulation_inventory_log"]

reproduction_check = sim_inventory.merge(
    resolved.observed_inventory.rename(columns={"quantity": "observed_qty"}),
    on=["period_id", "facility_id", "commodity_category"],
    how="inner",
)
reproduction_check["diff"] = (
    reproduction_check["quantity"] - reproduction_check["observed_qty"]
)
reproduction_max_abs_diff = reproduction_check["diff"].abs().max()
reproduction_mismatches = int((reproduction_check["diff"].abs() > 1e-9).sum())

assert reproduction_mismatches == 0, (
    f"simulator diverges from observed_inventory in {reproduction_mismatches} rows "
    f"(max abs diff {reproduction_max_abs_diff})"
)

## 3. Phase order matters — `[Demand, Arrivals]` overshoots

At daily grain the "correct" identity `inv_init + arrivals − demand = inv_end` only holds if arrivals are applied **before** demand. Running demand first causes `DemandPhase` to clip the subtraction at zero (logging "unmet" deficits) while `ArrivalsPhase` still adds the full return count afterwards, leaving inventory above the historical trace.

In [ ]:
bad_order_env = Environment(
    resolved,
    EnvironmentConfig(
        phases=[DemandPhase(), ArrivalsPhase()],
        seed=42,
        scenario_id="wrong_order",
    ),
)
bad_order_inventory = bad_order_env.run().to_dataframes()["simulation_inventory_log"]
bad_order_check = bad_order_inventory.merge(
    resolved.observed_inventory.rename(columns={"quantity": "observed_qty"}),
    on=["period_id", "facility_id", "commodity_category"],
    how="inner",
)
bad_order_check["diff"] = (
    bad_order_check["quantity"] - bad_order_check["observed_qty"]
)
bad_order_mismatches = int((bad_order_check["diff"].abs() > 1e-9).sum())
bad_order_max_overshoot = bad_order_check["diff"].max()

assert bad_order_mismatches > 0 and bad_order_max_overshoot > 0, (
    "expected [Demand, Arrivals] ordering to overshoot observed inventory"
)